# LeafDoctor: offline plant-disease checker (MAIB AI 221, Project 2)
**Run on a free Colab T4 GPU:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all (about 15 min).

What this notebook does, in order:
1. Downloads **PlantVillage** (lab photos, training) and **PlantDoc** (field photos, CC BY 4.0, test) and keeps the 28 classes they share.
2. **Baselines:** a classical colour-histogram classifier, and frozen ImageNet features with logistic regression.
3. **Model A:** MobileNetV3-Small, fine-tuned with augmentation on lab photos only.
4. **Generative AI:** SD-Turbo makes realistic field backgrounds (soil, hands, foliage, harsh light), lab leaves are placed on them, then a light image-to-image pass blends them in. **Model B** is trained on lab + synthetic images.
5. **Evaluation:** accuracy, macro-F1 and top-3 on lab test vs field test (the domain shift), temperature calibration, a reliability diagram, the "unsure" rejection threshold, a confusion matrix, per-crop results and failure examples.
6. **Export:** ONNX with Grad-CAM heat-map output, int8 quantisation, and a speed test.

Output: `leafdoctor_out.zip`. Put `model/` into the web app, and use `figures/` and `report_numbers.md` for the report.

In [ ]:
# 1 · Setup
import os, sys, subprocess, json, time, random, math, shutil
from pathlib import Path
LOCAL_TEST = os.environ.get("LEAFDOCTOR_LOCAL_TEST") == "1"   # tiny CPU dry run, used only to check the code
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "onnx", "onnxruntime"], check=True)

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps, ImageFilter
import cv2, matplotlib
if not IN_COLAB: matplotlib.use("Agg")
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, confusion_matrix
from scipy.ndimage import binary_fill_holes

SEED = 42
N_TRAIN, N_VAL, N_TEST = 120, 20, 40     # PlantVillage images per class
N_SYN = 20                               # diffusion images per class
EPOCHS, BATCH, LR, IMG, STORE = 4, 64, 1.5e-3, 224, 256
TARGET_ACC = 0.70                        # wanted accuracy on photos the app does NOT mark "unsure"
if LOCAL_TEST:
    N_TRAIN, N_VAL, N_TEST, N_SYN, EPOCHS, BATCH = 8, 3, 3, 2, 1, 16

ROOT = Path("/content") if IN_COLAB else Path(os.environ.get("LEAFDOCTOR_ROOT", "./run"))
DATA, OUT = ROOT / "data", ROOT / "leafdoctor_out"
FIG, MODEL_DIR = OUT / "figures", OUT / "model"
for d in (DATA, FIG, MODEL_DIR): d.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)
RESULTS = {"config": dict(N_TRAIN=N_TRAIN, N_VAL=N_VAL, N_TEST=N_TEST, N_SYN=N_SYN, EPOCHS=EPOCHS, BATCH=BATCH, LR=LR,
                          backbone="MobileNetV3-Small (ImageNet)", seed=SEED, target_acc=TARGET_ACC)}
def save_results(): (OUT / "results.json").write_text(json.dumps(RESULTS, indent=2))

In [ ]:
# 2 · Classes shared by PlantVillage and PlantDoc: (PlantVillage folder, PlantDoc folder, display name, crop)
CLASSES = [
 ("Apple___Apple_scab", "Apple Scab Leaf", "Apple scab", "Apple"),
 ("Apple___Cedar_apple_rust", "Apple rust leaf", "Apple cedar rust", "Apple"),
 ("Apple___healthy", "Apple leaf", "Apple (healthy)", "Apple"),
 ("Pepper,_bell___Bacterial_spot", "Bell_pepper leaf spot", "Bell pepper bacterial spot", "Bell pepper"),
 ("Pepper,_bell___healthy", "Bell_pepper leaf", "Bell pepper (healthy)", "Bell pepper"),
 ("Blueberry___healthy", "Blueberry leaf", "Blueberry (healthy)", "Blueberry"),
 ("Cherry_(including_sour)___healthy", "Cherry leaf", "Cherry (healthy)", "Cherry"),
 ("Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn Gray leaf spot", "Corn gray leaf spot", "Corn"),
 ("Corn_(maize)___Northern_Leaf_Blight", "Corn leaf blight", "Corn northern leaf blight", "Corn"),
 ("Corn_(maize)___Common_rust_", "Corn rust leaf", "Corn common rust", "Corn"),
 ("Grape___Black_rot", "grape leaf black rot", "Grape black rot", "Grape"),
 ("Grape___healthy", "grape leaf", "Grape (healthy)", "Grape"),
 ("Peach___healthy", "Peach leaf", "Peach (healthy)", "Peach"),
 ("Potato___Early_blight", "Potato leaf early blight", "Potato early blight", "Potato"),
 ("Potato___Late_blight", "Potato leaf late blight", "Potato late blight", "Potato"),
 ("Raspberry___healthy", "Raspberry leaf", "Raspberry (healthy)", "Raspberry"),
 ("Soybean___healthy", "Soyabean leaf", "Soybean (healthy)", "Soybean"),
 ("Squash___Powdery_mildew", "Squash Powdery mildew leaf", "Squash powdery mildew", "Squash"),
 ("Strawberry___healthy", "Strawberry leaf", "Strawberry (healthy)", "Strawberry"),
 ("Tomato___Bacterial_spot", "Tomato leaf bacterial spot", "Tomato bacterial spot", "Tomato"),
 ("Tomato___Early_blight", "Tomato Early blight leaf", "Tomato early blight", "Tomato"),
 ("Tomato___Late_blight", "Tomato leaf late blight", "Tomato late blight", "Tomato"),
 ("Tomato___Leaf_Mold", "Tomato mold leaf", "Tomato leaf mold", "Tomato"),
 ("Tomato___Septoria_leaf_spot", "Tomato Septoria leaf spot", "Tomato Septoria leaf spot", "Tomato"),
 ("Tomato___Spider_mites Two-spotted_spider_mite", "Tomato two spotted spider mites leaf", "Tomato spider mites", "Tomato"),
 ("Tomato___Tomato_mosaic_virus", "Tomato leaf mosaic virus", "Tomato mosaic virus", "Tomato"),
 ("Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato leaf yellow virus", "Tomato yellow leaf curl virus", "Tomato"),
 ("Tomato___healthy", "Tomato leaf", "Tomato (healthy)", "Tomato"),
]
NC = len(CLASSES); NAMES = [c[2] for c in CLASSES]; CROPS = [c[3] for c in CLASSES]

def sh(*a): subprocess.run([str(x) for x in a], check=True)
pv_dir, pd_dir = DATA / "PlantVillage-Dataset", DATA / "PlantDoc-Dataset"
t0 = time.time()
if not (pv_dir / "raw").exists():   # sparse checkout: only the 28 colour-photo folders we need
    sh("git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/spMohanty/PlantVillage-Dataset", pv_dir)
    sh("git", "-C", pv_dir, "sparse-checkout", "set", "--no-cone", *[f"/raw/color/{c[0]}/" for c in CLASSES])
if not (pd_dir / "test").exists():
    sh("git", "clone", "-q", "--depth", "1", "https://github.com/pratikkayal/PlantDoc-Dataset", pd_dir)
print(f"download done in {time.time()-t0:.0f}s")

In [ ]:
# 3 · Splits and loading. Every image is stored as a 256x256 centre square; evaluation takes the central 224 crop.
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def list_imgs(d):
    d = Path(d); return sorted(p for p in d.iterdir() if p.suffix.lower() in EXT) if d.exists() else []

def load_square(path, size=STORE):
    try:
        im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    except Exception:
        return None
    w, h = im.size; s = min(w, h); l, t = (w - s) // 2, (h - s) // 2
    return np.asarray(im.crop((l, t, l + s, t + s)).resize((size, size), Image.BILINEAR))

def load_set(items):
    with ThreadPoolExecutor(8) as ex: arrs = list(ex.map(lambda it: load_square(it[0]), items))
    keep = [i for i, a in enumerate(arrs) if a is not None]
    return (np.stack([arrs[i] for i in keep]), np.array([items[i][1] for i in keep]), [str(items[i][0]) for i in keep])

rng = random.Random(SEED)
pv_items = {"train": [], "val": [], "test": []}
for k, c in enumerate(CLASSES):
    files = list_imgs(pv_dir / "raw" / "color" / c[0]); rng.shuffle(files)
    n = len(files); nt, nv = min(N_TEST, n // 5), min(N_VAL, n // 10)
    pv_items["test"] += [(f, k) for f in files[:nt]]
    pv_items["val"] += [(f, k) for f in files[nt:nt + nv]]
    pv_items["train"] += [(f, k) for f in files[nt + nv:nt + nv + N_TRAIN]]
field_val_items = [(f, k) for k, c in enumerate(CLASSES) for f in list_imgs(pd_dir / "train" / c[1])]   # field calibration set
field_test_items = [(f, k) for k, c in enumerate(CLASSES) for f in list_imgs(pd_dir / "test" / c[1])]  # field test set (never used to fit anything)
if LOCAL_TEST:
    field_val_items = rng.sample(field_val_items, 60); field_test_items = rng.sample(field_test_items, 40)

X_tr, y_tr, _ = load_set(pv_items["train"]); X_va, y_va, _ = load_set(pv_items["val"]); X_te, y_te, _ = load_set(pv_items["test"])
X_fv, y_fv, _ = load_set(field_val_items); X_ft, y_ft, P_ft = load_set(field_test_items)
RESULTS["data"] = {"plantvillage_train": len(y_tr), "plantvillage_val": len(y_va), "plantvillage_test": len(y_te),
                   "plantdoc_field_val(train folder)": len(y_fv), "plantdoc_field_test(test folder)": len(y_ft), "classes": NC,
                   "field_test_per_class": {NAMES[k]: int((y_ft == k).sum()) for k in range(NC)}}
print(json.dumps(RESULTS["data"], indent=1)); save_results()

fig, ax = plt.subplots(2, 6, figsize=(15, 5.5))
for i in range(6):
    j = np.random.randint(len(y_tr)); ax[0, i].imshow(X_tr[j]); ax[0, i].set_title("LAB: " + NAMES[y_tr[j]], fontsize=7)
    j = np.random.randint(len(y_ft)); ax[1, i].imshow(X_ft[j]); ax[1, i].set_title("FIELD: " + NAMES[y_ft[j]], fontsize=7)
for a in ax.flat: a.axis("off")
plt.suptitle("Domain shift: PlantVillage lab photos (top) vs PlantDoc field photos (bottom)"); plt.tight_layout()
plt.savefig(FIG / "01_lab_vs_field.png", dpi=110); plt.show()

In [ ]:
# 4 · Model, training and evaluation helpers
class LeafNet(nn.Module):
    """MobileNetV3-Small with ImageNet normalisation inside, so the input is RGB in [0,1]."""
    def __init__(self):
        super().__init__()
        w = None if LOCAL_TEST else torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        self.m = torchvision.models.mobilenet_v3_small(weights=w)
        self.m.classifier[3] = nn.Linear(self.m.classifier[3].in_features, NC)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x): return self.m((x - self.mean) / self.std)

train_tf = T.Compose([T.RandomResizedCrop(IMG, scale=(0.45, 1.0), antialias=True), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomApply([T.RandomRotation(25)], p=0.5), T.ColorJitter(0.35, 0.35, 0.35, 0.04),
                      T.RandomApply([T.GaussianBlur(5, (0.1, 1.5))], p=0.2)])
eval_tf = T.CenterCrop(IMG)

class ArrDS(Dataset):
    def __init__(self, X, y, tf): self.X, self.y, self.tf = X, y, tf
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.tf(torch.from_numpy(self.X[i]).permute(2, 0, 1)).float().div(255), int(self.y[i])

AMP = device == "cuda"
@torch.no_grad()
def get_logits(model, X, bs=128):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = eval_tf(torch.from_numpy(X[i:i + bs]).permute(0, 3, 1, 2)).float().div(255).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): out.append(model(x).float().cpu())
    return torch.cat(out).numpy()

def metrics(logits, y):
    p = logits.argmax(1); top3 = np.argsort(-logits, 1)[:, :3]
    return {"acc": round(float((p == y).mean()), 4),
            "macro_f1": round(float(f1_score(y, p, labels=np.unique(y), average="macro", zero_division=0)), 4),
            "top3_acc": round(float((top3 == y[:, None]).any(1).mean()), 4), "n": int(len(y))}

def train_model(X, y, name):
    torch.manual_seed(SEED)
    model = LeafNet().to(device)
    dl = DataLoader(ArrDS(X, y, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, pin_memory=AMP)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.2)
    scaler = torch.amp.GradScaler(enabled=AMP); crit = nn.CrossEntropyLoss(label_smoothing=0.1); hist = []
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tot = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): loss = crit(model(xb), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        h = {"epoch": ep + 1, "loss": round(tot / len(dl), 4), "lab_val_acc": metrics(get_logits(model, X_va), y_va)["acc"],
             "field_val_acc": metrics(get_logits(model, X_fv), y_fv)["acc"]}
        hist.append(h); print(name, h, f"{time.time()-t0:.0f}s")
    return model, hist

In [ ]:
# 5 · Baselines: (B0) classical HSV colour histogram + logistic regression, (B1) frozen ImageNet MobileNetV3-Small + logistic regression
def hsv_hist(X):
    feats = []
    for a in X:
        hsv = cv2.cvtColor(a[16:240, 16:240], cv2.COLOR_RGB2HSV)
        h = cv2.calcHist([hsv], [0, 1, 2], None, [16, 4, 4], [0, 180, 0, 256, 0, 256]).flatten()
        feats.append(h / (h.sum() + 1e-6))
    return np.array(feats)

@torch.no_grad()
def frozen_feats(X, net):
    net.eval(); out = []
    for i in range(0, len(X), 128):
        x = eval_tf(torch.from_numpy(X[i:i + 128]).permute(0, 3, 1, 2)).float().div(255).to(device)
        x = (x - net.mean) / net.std
        out.append(net.m.avgpool(net.m.features(x)).flatten(1).float().cpu())
    return torch.cat(out).numpy()

def lr_logits(clf, F_): return clf.decision_function(F_)
RESULTS["baselines"] = {}
clf0 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(hsv_hist(X_tr), y_tr)
RESULTS["baselines"]["B0_hsv_histogram_logreg"] = {"lab_test": metrics(lr_logits(clf0, hsv_hist(X_te)), y_te),
                                                   "field_test": metrics(lr_logits(clf0, hsv_hist(X_ft)), y_ft)}
frozen = LeafNet().to(device)
clf1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(frozen_feats(X_tr, frozen), y_tr)
RESULTS["baselines"]["B1_frozen_imagenet_logreg"] = {"lab_test": metrics(lr_logits(clf1, frozen_feats(X_te, frozen)), y_te),
                                                     "field_test": metrics(lr_logits(clf1, frozen_feats(X_ft, frozen)), y_ft)}
del frozen; print(json.dumps(RESULTS["baselines"], indent=1)); save_results()

In [ ]:
# 6 · Model A: fine-tuned on lab photos only
model_A, hist_A = train_model(X_tr, y_tr, "A(lab only)")
RESULTS["model_A_lab_only"] = {"history": hist_A, "lab_test": metrics(get_logits(model_A, X_te), y_te),
                               "field_test": metrics(get_logits(model_A, X_ft), y_ft)}
print(RESULTS["model_A_lab_only"]["lab_test"], RESULTS["model_A_lab_only"]["field_test"]); save_results()

In [ ]:
# 7 · Generative AI: diffusion-made field backgrounds + lab leaf compositing + SD-Turbo image-to-image harmonisation
BG_PROMPTS = [
 "close-up photo of dry brown soil in a farm field, harsh midday sunlight, hard shadows",
 "a farmer's open hand palm outdoors in a crop field, natural daylight, smartphone photo",
 "dense green crop foliage, blurry background, smartphone photo in a vegetable field",
 "vegetable garden bed with mulch, weeds and drip irrigation pipe, overcast light",
 "tomato plants in a greenhouse, blurry green background, warm light",
 "dusty ground with dry grass and small stones, strong sun, high contrast",
 "rows of young plants in a field, early morning light, slightly out of focus",
 "dark wet soil after rain, scattered fallen leaves, cloudy day",
 "farm worker's gloved hand outdoors, green leaves behind, bright sunlight",
 "potato field with green plants and soil furrows, afternoon sun",
]
def leaf_mask(a):
    """Leaf = green/yellow/brown hues with some saturation; the grey-purple lab background and its shadows are excluded."""
    hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV); H, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    m = (((H < 95) | (H > 170)) & (S > 30) & (V > 25)).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8), iterations=2)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m)
    if n < 2: return None
    m = binary_fill_holes(lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    return m if 0.08 < m.mean() < 0.92 else None

def composite(leaf, mask, bg, r, S=512):
    size = int(S * r.uniform(0.55, 0.9)); ang = r.uniform(-40, 40)
    li = Image.fromarray(leaf).resize((size, size), Image.BILINEAR).rotate(ang, resample=Image.BILINEAR)
    mi = Image.fromarray(mask * 255).resize((size, size), Image.BILINEAR).rotate(ang, resample=Image.BILINEAR).filter(ImageFilter.GaussianBlur(2))
    x, y = r.randint(0, S - size), r.randint(0, S - size)
    canvas = Image.fromarray(bg).resize((S, S)).copy(); alpha = Image.new("L", (S, S), 0)
    canvas.paste(li, (x, y), mi); alpha.paste(mi, (x, y))
    return np.asarray(canvas), np.asarray(alpha).astype(np.float32)[..., None] / 255

def harsh_light(a, r):
    a = (a.astype(np.float32) / 255) ** r.uniform(0.7, 1.4) * np.array([r.uniform(0.85, 1.15) for _ in range(3)])
    if r.random() < 0.5:
        h, w = a.shape[:2]; yy, xx = np.mgrid[0:h, 0:w]; cy, cx, rad = r.uniform(0, h), r.uniform(0, w), r.uniform(0.3, 0.8) * w
        a = a + r.uniform(0.1, 0.35) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * rad * rad))[..., None]
    return (np.clip(a, 0, 1) * 255).astype(np.uint8)

t0 = time.time(); r = random.Random(SEED)
if LOCAL_TEST:   # no GPU or Hugging Face here: plain noise backgrounds stand in, just to test the code path
    bgs = [(np.random.rand(512, 512, 3) * 255).astype(np.uint8) for _ in range(4)]; bg_prompt_idx = [0] * 4
    harmonise = lambda imgs, prompts: imgs
else:
    from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image
    t2i = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
    t2i.set_progress_bar_config(disable=True)
    i2i = AutoPipelineForImage2Image.from_pipe(t2i)
    bgs, bg_prompt_idx = [], []
    for b in range(8):   # 8 batches x 8 = 64 backgrounds, about 1 step each
        idx = [(b * 8 + j) % len(BG_PROMPTS) for j in range(8)]
        ims = t2i(prompt=[BG_PROMPTS[i] for i in idx], num_inference_steps=1, guidance_scale=0.0, height=512, width=512,
                  generator=torch.Generator("cuda").manual_seed(SEED + b)).images
        bgs += [np.asarray(im) for im in ims]; bg_prompt_idx += idx
    def harmonise(imgs, prompts):
        out = i2i(prompt=prompts, image=[Image.fromarray(i) for i in imgs], strength=0.35, num_inference_steps=3, guidance_scale=0.0).images
        return [np.asarray(o) for o in out]

syn_X, syn_y, examples = [], [], []
for k in range(NC):
    src = [i for i in np.where(y_tr == k)[0]]; r.shuffle(src); made = 0
    batch_c, batch_a, batch_p, batch_src = [], [], [], []
    for i in src:
        if made + len(batch_c) >= N_SYN: break
        m = leaf_mask(X_tr[i])
        if m is None: continue
        bi = r.randrange(len(bgs)); comp, alpha = composite(X_tr[i], m, bgs[bi], r)
        batch_c.append(comp); batch_a.append(alpha); batch_src.append(i)
        batch_p.append("photo of a diseased plant leaf, " + BG_PROMPTS[bg_prompt_idx[bi]])
    for j in range(0, len(batch_c), 8):
        harm = harmonise(batch_c[j:j + 8], batch_p[j:j + 8])
        for comp, alpha, h, si in zip(batch_c[j:j + 8], batch_a[j:j + 8], harm, batch_src[j:j + 8]):
            keep = alpha * 0.8   # keep 80% of the original leaf pixels so the disease lesions are preserved
            final = harsh_light((keep * comp + (1 - keep) * h).astype(np.uint8), r)
            syn_X.append(np.asarray(Image.fromarray(final).resize((STORE, STORE), Image.BILINEAR))); syn_y.append(k)
            if len(examples) < 8 and made == 0: examples.append((X_tr[si], final, NAMES[k]))
            made += 1
syn_X, syn_y = np.stack(syn_X), np.array(syn_y)
RESULTS["generative"] = {"model": "stabilityai/sd-turbo (text-to-image backgrounds + image-to-image harmonisation, strength 0.35)",
                         "n_backgrounds": len(bgs), "n_synthetic": int(len(syn_y)), "seconds": round(time.time() - t0, 1)}
print(RESULTS["generative"]); save_results()
if not LOCAL_TEST:
    del t2i, i2i; torch.cuda.empty_cache()

fig, ax = plt.subplots(2, len(examples), figsize=(2.2 * len(examples), 5))
for j, (a, b, n) in enumerate(examples):
    ax[0, j].imshow(a); ax[0, j].set_title(n, fontsize=7); ax[1, j].imshow(b)
for a in ax.flat: a.axis("off")
plt.suptitle("Lab leaf (top) and diffusion-made field version (bottom)"); plt.tight_layout()
plt.savefig(FIG / "02_synthetic_examples.png", dpi=110); plt.show()
nb = min(6, len(bgs)); fig, ax = plt.subplots(1, nb, figsize=(14, 2.6))
for j in range(nb): ax[j].imshow(bgs[j]); ax[j].axis("off")
plt.suptitle("SD-Turbo generated field backgrounds"); plt.tight_layout(); plt.savefig(FIG / "03_backgrounds.png", dpi=100); plt.show()

In [ ]:
# 8 · Model B: lab photos + diffusion-augmented field-style images
model_B, hist_B = train_model(np.concatenate([X_tr, syn_X]), np.concatenate([y_tr, syn_y]), "B(lab+diffusion)")
RESULTS["model_B_lab_plus_diffusion"] = {"history": hist_B, "lab_test": metrics(get_logits(model_B, X_te), y_te),
                                         "field_test": metrics(get_logits(model_B, X_ft), y_ft)}
print(RESULTS["model_B_lab_plus_diffusion"]["lab_test"], RESULTS["model_B_lab_plus_diffusion"]["field_test"]); save_results()

# Choose the deployed model on the FIELD VALIDATION set (PlantDoc train folder), never on the test set.
fv_A, fv_B = metrics(get_logits(model_A, X_fv), y_fv)["acc"], metrics(get_logits(model_B, X_fv), y_fv)["acc"]
best_name, model = ("B", model_B) if fv_B >= fv_A else ("A", model_A)
RESULTS["deployed_model"] = {"choice": best_name, "field_val_acc_A": fv_A, "field_val_acc_B": fv_B}
print("deploying model", best_name, RESULTS["deployed_model"]); save_results()

In [ ]:
# 9 · Calibration (temperature scaling on field validation), reliability diagram and the "unsure" threshold
def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def fit_temperature(logits, y):
    lt = torch.zeros(1, requires_grad=True); L, Y = torch.tensor(logits), torch.tensor(y)
    opt = torch.optim.LBFGS([lt], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(L / lt.exp(), Y); loss.backward(); return loss
    opt.step(closure); return float(lt.exp())
def ece(p, y, bins=10):
    conf, pred = p.max(1), p.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return round(float(e), 4)

L_fv, L_ft, L_te = get_logits(model, X_fv), get_logits(model, X_ft), get_logits(model, X_te)
TEMP = fit_temperature(L_fv, y_fv)
P_ft_raw, P_ft = softmax(L_ft), softmax(L_ft / TEMP)

conf_v, corr_v = softmax(L_fv / TEMP).max(1), softmax(L_fv / TEMP).argmax(1) == y_fv
TAU = None
for tau in np.linspace(0.05, 0.95, 91):
    acc_m = conf_v >= tau
    if acc_m.mean() >= 0.05 and corr_v[acc_m].mean() >= TARGET_ACC: TAU = float(tau); break
if TAU is None: TAU = float(np.quantile(conf_v, 0.5))   # fallback: mark the least confident half "unsure"
acc_mask = P_ft.max(1) >= TAU
RESULTS["calibration"] = {"temperature": round(TEMP, 4), "ece_field_test_before": ece(P_ft_raw, y_ft), "ece_field_test_after": ece(P_ft, y_ft),
                          "ece_lab_test_after": ece(softmax(L_te / TEMP), y_te)}
RESULTS["rejection"] = {"threshold": round(TAU, 3), "target_acc_on_accepted(field_val)": TARGET_ACC,
                        "field_test_share_unsure": round(float(1 - acc_mask.mean()), 4),
                        "field_test_acc_on_accepted": round(float((P_ft.argmax(1) == y_ft)[acc_mask].mean()), 4) if acc_mask.any() else None,
                        "field_test_acc_all": round(float((P_ft.argmax(1) == y_ft).mean()), 4)}
print(RESULTS["calibration"], RESULTS["rejection"]); save_results()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for P, lab in ((P_ft_raw, "before (T=1)"), (P_ft, f"after (T={TEMP:.2f})")):
    conf, cor = P.max(1), P.argmax(1) == y_ft; xs, ys = [], []
    for lo in np.linspace(0, 1, 11)[:-1]:
        m = (conf > lo) & (conf <= lo + 0.1)
        if m.sum() >= 3: xs.append(conf[m].mean()); ys.append(cor[m].mean())
    ax[0].plot(xs, ys, "o-", label=f"{lab}, ECE={ece(P, y_ft):.3f}")
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set_xlabel("confidence"); ax[0].set_ylabel("accuracy"); ax[0].set_title("Reliability diagram (field test)"); ax[0].legend()
taus = np.linspace(0, 0.95, 60); cov = [(P_ft.max(1) >= t).mean() for t in taus]
accs = [(P_ft.argmax(1) == y_ft)[P_ft.max(1) >= t].mean() if (P_ft.max(1) >= t).any() else np.nan for t in taus]
ax[1].plot(cov, accs, "-"); ax[1].axvline(acc_mask.mean(), color="r", ls="--", label=f"threshold {TAU:.2f}")
ax[1].set_xlabel("coverage (share of photos answered)"); ax[1].set_ylabel("accuracy on answered photos"); ax[1].set_title("Accuracy vs coverage (field test)"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / "04_calibration_rejection.png", dpi=110); plt.show()

In [ ]:
# 10 · Error analysis: confusion matrix, per-crop (subgroup) results, per-class domain shift, failure gallery
pred_ft = P_ft.argmax(1)
cm = confusion_matrix(y_ft, pred_ft, labels=range(NC)); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
plt.figure(figsize=(11, 9.5)); plt.imshow(cmn, cmap="Blues"); plt.colorbar(fraction=0.04)
plt.xticks(range(NC), NAMES, rotation=90, fontsize=7); plt.yticks(range(NC), NAMES, fontsize=7)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title(f"Confusion matrix, field test (model {best_name}, row-normalised)")
plt.tight_layout(); plt.savefig(FIG / "05_confusion_field.png", dpi=110); plt.show()

pairs = sorted(((cm[i, j], NAMES[i], NAMES[j]) for i in range(NC) for j in range(NC) if i != j and cm[i, j] > 0), reverse=True)[:8]
RESULTS["top_confusions_field"] = [{"true": a, "pred": b, "count": int(n)} for n, a, b in pairs]
crop_res = {}
for crop in sorted(set(CROPS)):
    m = np.array([CROPS[k] == crop for k in y_ft])
    if m.any(): crop_res[crop] = {"n": int(m.sum()), "acc": round(float((pred_ft[m] == y_ft[m]).mean()), 3),
                                  "unsure_share": round(float(1 - acc_mask[m].mean()), 3)}
RESULTS["per_crop_field"] = crop_res
pred_te = L_te.argmax(1); per_cls = {}
for k in range(NC):
    a = (pred_te[y_te == k] == k).mean() if (y_te == k).any() else np.nan
    b = (pred_ft[y_ft == k] == k).mean() if (y_ft == k).any() else np.nan
    per_cls[NAMES[k]] = {"lab_acc": None if np.isnan(a) else round(float(a), 3), "field_acc": None if np.isnan(b) else round(float(b), 3)}
RESULTS["per_class_lab_vs_field"] = per_cls; save_results()

plt.figure(figsize=(12, 4.5)); xs = np.arange(NC)
plt.bar(xs - 0.2, [per_cls[n]["lab_acc"] or 0 for n in NAMES], 0.4, label="lab test (PlantVillage)")
plt.bar(xs + 0.2, [per_cls[n]["field_acc"] or 0 for n in NAMES], 0.4, label="field test (PlantDoc)")
plt.xticks(xs, NAMES, rotation=90, fontsize=7); plt.ylabel("accuracy"); plt.title("Per-class domain shift"); plt.legend(); plt.tight_layout()
plt.savefig(FIG / "06_per_class_shift.png", dpi=110); plt.show()

wrong = np.where(pred_ft != y_ft)[0]; wrong = wrong[np.argsort(-P_ft[wrong].max(1))][:12]
if len(wrong):
    fig, ax = plt.subplots(2, 6, figsize=(15, 6))
    for a, i in zip(ax.flat, wrong):
        a.imshow(X_ft[i][16:240, 16:240]); a.set_title(f"true: {NAMES[y_ft[i]]}\npred: {NAMES[pred_ft[i]]} ({P_ft[i].max():.0%})", fontsize=7)
    for a in ax.flat: a.axis("off")
    plt.suptitle("Most confident mistakes on field photos"); plt.tight_layout(); plt.savefig(FIG / "07_failures.png", dpi=110); plt.show()

In [ ]:
# 11 · Export to ONNX with a Grad-CAM output (computed inside the graph), int8 quantisation, speed test
class Deploy(nn.Module):
    """Outputs logits and a 7x7 Grad-CAM map for the top class. The gradient of the MLP head is written out by hand, so no autograd is needed at run time."""
    def __init__(self, net):
        super().__init__(); m = net.m
        self.features, self.fc1, self.fc2 = m.features, m.classifier[0], m.classifier[3]
        self.register_buffer("mean", net.mean.clone()); self.register_buffer("std", net.std.clone())
    def forward(self, x):
        A = self.features((x - self.mean) / self.std)              # [B,576,7,7]
        z = self.fc1(A.mean((2, 3)))                               # [B,1024]
        logits = self.fc2(F.hardswish(z))                          # [B,NC]
        w2 = self.fc2.weight[logits.argmax(1)]                     # [B,1024]
        dz = torch.where(z < -3, torch.zeros_like(z), torch.where(z > 3, torch.ones_like(z), (2 * z + 3) / 6))  # hardswish'
        g = torch.matmul(w2 * dz, self.fc1.weight)                 # d logit / d pooled feature, [B,576]
        cam = F.relu((g[:, :, None, None] * A).sum(1))             # [B,7,7]
        return logits, cam

dep = Deploy(model.float().cpu()).eval()
with torch.no_grad():   # check the deploy wrapper gives the same logits as the trained model
    x = eval_tf(torch.from_numpy(X_ft[:1]).permute(0, 3, 1, 2)).float().div(255).requires_grad_(False)
    lg, cam = dep(x); assert torch.allclose(lg, model(x), atol=1e-4)
fp32_path, pre_path, int8_path = OUT / "model_fp32.onnx", OUT / "model_pre.onnx", OUT / "model_int8.onnx"
kw = dict(input_names=["image"], output_names=["logits", "cam"], dynamic_axes={"image": {0: "b"}, "logits": {0: "b"}, "cam": {0: "b"}}, opset_version=17)
try: torch.onnx.export(dep, torch.zeros(1, 3, IMG, IMG), str(fp32_path), dynamo=False, **kw)
except TypeError: torch.onnx.export(dep, torch.zeros(1, 3, IMG, IMG), str(fp32_path), **kw)

import onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
quant_pre_process(str(fp32_path), str(pre_path))
def to_input(X): return eval_tf(torch.from_numpy(X).permute(0, 3, 1, 2)).float().div(255).numpy()
class Reader(CalibrationDataReader):
    def __init__(self):
        idx = np.random.RandomState(0).permutation(len(X_fv))[:64]; self.data = iter([{"image": to_input(X_fv[i:i + 1])} for i in idx] +
                                                                                   [{"image": to_input(X_tr[i:i + 1])} for i in range(0, len(X_tr), max(1, len(X_tr) // 64))])
    def get_next(self): return next(self.data, None)
quantize_static(str(pre_path), str(int8_path), Reader(), quant_format=QuantFormat.QDQ, per_channel=True,
                weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8, op_types_to_quantize=["Conv"])

def onnx_eval(path, X, y):
    s = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"]); L = []
    for i in range(0, len(X), 32): L.append(s.run(["logits"], {"image": to_input(X[i:i + 32])})[0])
    L = np.concatenate(L); so = ort.SessionOptions(); so.intra_op_num_threads = 1
    s1 = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"]); one = to_input(X[:1]); ts = []
    for _ in range(30): t = time.perf_counter(); s1.run(None, {"image": one}); ts.append(time.perf_counter() - t)
    return metrics(L, y), round(float(np.median(ts) * 1000), 1)
q = {}
for nm, pth in (("fp32", fp32_path), ("int8", int8_path)):
    mf, ms = onnx_eval(pth, X_ft, y_ft); ml, _ = onnx_eval(pth, X_te, y_te)
    q[nm] = {"size_MB": round(pth.stat().st_size / 1e6, 2), "field_test": mf, "lab_test": ml, "cpu_1thread_ms": ms}
use = "int8" if q["int8"]["field_test"]["acc"] >= q["fp32"]["field_test"]["acc"] - 0.02 else "fp32"
shutil.copy(int8_path if use == "int8" else fp32_path, MODEL_DIR / "model.onnx")
RESULTS["export"] = {**q, "deployed_variant": use}; print(json.dumps(RESULTS["export"], indent=1)); save_results()

In [ ]:
# 12 · Config for the web app, a summary for the report, and the zip
config = {"version": time.strftime("%Y%m%d-%H%M"), "input_size": IMG, "crop_fraction": IMG / STORE, "temperature": round(TEMP, 4),
          "threshold": round(TAU, 3), "model_variant": use, "trained_model": best_name,
          "classes": [{"id": c[0], "name": c[2], "crop": c[3]} for c in CLASSES]}
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=1))
torch.save(model.state_dict(), OUT / "leafnet_best.pt")

def row(name, d): return f"| {name} | {d['lab_test']['acc']:.3f} | {d['lab_test']['macro_f1']:.3f} | {d['field_test']['acc']:.3f} | {d['field_test']['macro_f1']:.3f} | {d['field_test']['top3_acc']:.3f} | {d['lab_test']['acc']-d['field_test']['acc']:+.3f} |"
R = RESULTS; md = ["# LeafDoctor results (auto-generated)", "", "## Lab vs field (domain shift)", "",
      "| Model | Lab acc | Lab macro-F1 | Field acc | Field macro-F1 | Field top-3 | Drop (lab-field) |", "|---|---|---|---|---|---|---|",
      row("B0 classical HSV histogram + LR", R["baselines"]["B0_hsv_histogram_logreg"]), row("B1 frozen ImageNet MobileNetV3-S + LR", R["baselines"]["B1_frozen_imagenet_logreg"]),
      row("A fine-tuned, lab only", R["model_A_lab_only"]), row("B fine-tuned, lab + diffusion", R["model_B_lab_plus_diffusion"]), "",
      f"Deployed: model {best_name} ({use}), chosen on field validation accuracy (A {fv_A:.3f} vs B {fv_B:.3f}).", "",
      "## Calibration and rejection (field test)", "", f"- Temperature T = {TEMP:.3f}; ECE {R['calibration']['ece_field_test_before']:.3f} → {R['calibration']['ece_field_test_after']:.3f}",
      f"- Threshold {TAU:.2f}: {R['rejection']['field_test_share_unsure']:.1%} of field photos marked unsure; accuracy on the rest {R['rejection']['field_test_acc_on_accepted']}",
      "", "## Export", "", f"- fp32 {q['fp32']['size_MB']} MB, field acc {q['fp32']['field_test']['acc']:.3f}, {q['fp32']['cpu_1thread_ms']} ms/image (1 CPU thread)",
      f"- int8 {q['int8']['size_MB']} MB, field acc {q['int8']['field_test']['acc']:.3f}, {q['int8']['cpu_1thread_ms']} ms/image (1 CPU thread)", "",
      "## Per crop (field test, deployed model)", "", "| Crop | n | Acc | Unsure share |", "|---|---|---|---|"] + \
     [f"| {c} | {v['n']} | {v['acc']:.3f} | {v['unsure_share']:.1%} |" for c, v in crop_res.items()] + \
     ["", "## Top confusions (field test)", ""] + [f"- {p['true']} → {p['pred']}: {p['count']}" for p in R["top_confusions_field"]] + \
     ["", f"## Data\n\n{json.dumps(R['data'], indent=1)}", "", f"Generative: {json.dumps(R['generative'])}"]
(OUT / "report_numbers.md").write_text("\n".join(md)); print("\n".join(md))
shutil.make_archive(str(ROOT / "leafdoctor_out"), "zip", OUT)
if IN_COLAB:
    from google.colab import files; files.download(str(ROOT / "leafdoctor_out.zip"))